# Preprocess — DB → Parquet

**Run cells top-to-bottom once.**  
- Cell 1 installs dependencies  
- Cell 2 writes `worker.py` next to this notebook  
- Cell 3 lists the HuggingFace DB files  
- Cell 4 runs the processing loop  

In [ ]:
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade",
     "pyarrow", "pandas", "huggingface_hub", "tqdm"],
    check=True
)
print("Dependencies OK")

In [ ]:
import os, pathlib

WORKER_PATH = pathlib.Path("worker.py").resolve()

WORKER_CODE = '''
"""
worker.py  — spawned once per DB file by preprocess.ipynb.
Exits cleanly so the OS reclaims all memory unconditionally.
"""
import sys, os, gc, ctypes, sqlite3, shutil
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download


# ── helpers ──────────────────────────────────────────────────────────────────

def safe_int(s, default=0):
    return pd.to_numeric(s, errors="coerce").fillna(default).astype("Int64")

def safe_float(s, default=0.0):
    return pd.to_numeric(s, errors="coerce").fillna(default).astype(float)

def safe_str(s, default=""):
    return s.fillna(default).astype(str)


INT_COLS = [
    "message_id", "edit_hide", "noforwards", "scheduled", "pinned",
    "views", "forwards", "replies", "ttl", "forward", "reply",
    "reply_scheduled", "chain_imported", "media_price",
    "web_preview", "story", "photo", "video", "round", "voice",
    "other_media", "media_ttl", "buttons", "political",
]
FLOAT_COLS = [
    "toxicity", "severe_toxicity", "identity_attack",
    "insult", "profanity", "threat",
]
STR_COLS = [
    "content", "language", "from_id", "post_author", "date", "edit_date",
    "reply_source", "chain_from_id", "chain_from_name", "chain_post_author",
    "chain_date", "invoice", "contact", "location", "emails", "phones",
    "hashtags", "cashtags", "cards", "urls", "cleaner_urls",
    "domains", "identifiers", "restriction_reasons",
]
ALL_COLS = list(set(INT_COLS + FLOAT_COLS + STR_COLS))


def normalize(df, channel_name, source_db):
    for col in ALL_COLS:
        if col not in df.columns:
            df[col] = None
    for col in INT_COLS:   df[col] = safe_int(df[col])
    for col in FLOAT_COLS: df[col] = safe_float(df[col])
    for col in STR_COLS:   df[col] = safe_str(df[col])
    df["channel_name"] = channel_name
    df["source_db"]    = source_db
    return df


def filter_rows(df):
    has_content = df["content"].str.strip().str.len() > 0
    has_network = (
        (df["forward"]  > 0) | (df["forwards"] > 0) |
        (df["views"]    > 0) | (df["replies"]  > 0) | (df["reply"] > 0) |
        (df["chain_from_id"].str.strip().str.len() > 0) |
        (df["reply_source"].str.strip().str.len()  > 0)
    )
    has_media = (
        (df["photo"]       > 0) | (df["video"]       > 0) |
        (df["round"]       > 0) | (df["voice"]       > 0) |
        (df["story"]       > 0) | (df["other_media"] > 0) |
        (df["web_preview"] > 0) | (df["buttons"]     > 0)
    )
    return df[has_content | has_network | has_media].copy()


def release_memory():
    """Best-effort: return freed pages to the OS after each chunk."""
    gc.collect()
    try:
        pa.default_memory_pool().release_unused()   # correct PyArrow API name
    except AttributeError:
        pass
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)    # trim glibc arena on Linux
    except OSError:
        pass


# ── main ─────────────────────────────────────────────────────────────────────

def main():
    if len(sys.argv) != 4:
        print("Usage: worker.py <db_filename> <output_dir> <repo_id>", file=sys.stderr)
        sys.exit(1)

    db_filename, output_dir, repo_id = sys.argv[1], sys.argv[2], sys.argv[3]

    local_db = hf_hub_download(
        repo_id=repo_id, filename=db_filename,
        repo_type="dataset", revision="main",
    )

    channel_name = os.path.basename(local_db).replace(".db", "")
    parquet_out  = os.path.join(output_dir, f"{channel_name}.parquet")

    conn = sqlite3.connect(local_db)
    # FIX: disable SQLite page-cache + mmap.
    # Without these two pragmas, SQLite retains proportionally large RAM
    # even after conn.close() — empirically ~2x the DB size on a 1.5 GB file.
    conn.execute("PRAGMA cache_size = 0")
    conn.execute("PRAGMA mmap_size  = 0")

    writer = None
    try:
        for chunk in pd.read_sql_query("SELECT * FROM messages", conn, chunksize=100_000):
            chunk = normalize(chunk, channel_name, local_db)
            chunk = filter_rows(chunk)
            if len(chunk) == 0:
                del chunk; continue

            table = pa.Table.from_pandas(chunk, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(parquet_out, table.schema)
            writer.write_table(table)

            del chunk, table
            release_memory()

    finally:
        if writer: writer.close()
        conn.close()

    # Remove downloaded .db to free disk space
    try:    os.remove(local_db)
    except: pass

    # Clear HF cache — use HF_HOME so we delete the right directory
    cache = os.environ.get("HF_HOME", os.path.expanduser("~/.cache/huggingface"))
    if os.path.exists(cache):
        shutil.rmtree(cache, ignore_errors=True)


if __name__ == "__main__":
    main()
'''

WORKER_PATH.write_text(WORKER_CODE.lstrip())
print(f"worker.py written to: {WORKER_PATH}")

In [ ]:
import os
from huggingface_hub import list_repo_files

os.environ["HF_HOME"] = "/tmp/hf_cache"

OUTPUT_DIR = "./channels_20_parquet"
REPO_ID    = "leonardoblas/us_election_2024_telegram_distilled"
SUBFOLDER  = "channels_20"

os.makedirs(OUTPUT_DIR, exist_ok=True)

all_files    = list_repo_files(REPO_ID, repo_type="dataset")
db_filenames = [
    f for f in all_files
    if f.startswith(SUBFOLDER + "/") and f.endswith(".db")
]

print(f"Found {len(db_filenames)} DB files.")

In [ ]:
import sys, subprocess, os
from tqdm import tqdm

env = os.environ.copy()  # forwards HF_HOME to each worker

for db_file in tqdm(db_filenames, desc="Processing"):
    channel_name = os.path.basename(db_file).replace(".db", "")
    parquet_file = os.path.join(OUTPUT_DIR, f"{channel_name}.parquet")

    if os.path.exists(parquet_file):
        print(f"Skipping {channel_name} — already processed.")
        continue

    result = subprocess.run(
        [sys.executable, str(WORKER_PATH), db_file, OUTPUT_DIR, REPO_ID],
        env=env,
        stdout=None,   # stream output directly to notebook
        stderr=None,
    )

    if result.returncode != 0:
        print(f"[WARN] {channel_name} failed (exit {result.returncode}) — skipping.")